# Fiddler Simple Monitoring Quick Start Guide

## Goal

This guide provides a comprehensive tour of Fiddler's monitoring capabilities, demonstrating how to onboard a binary classification model and then layer on advanced features like proactive alerts, custom business metrics, user-defined segments, and multiple baseline strategies. 🗺️

## Deep Dive Into Fiddler's Monitoring and Analytic Capabilities

This notebook serves as a comprehensive introduction to Fiddler's powerful monitoring toolkit, using a customer churn model as the example. After walking through the fundamental steps of onboarding a binary classification model, the guide demonstrates how to build a robust, production-grade monitoring setup. You'll learn how to configure proactive alerts for data quality and performance, translate model metrics into business impact with custom metrics, isolate specific cohorts with segments, and implement both static and rolling baselines for powerful drift analysis. This guide showcases how these features work together to provide a complete view of your model's health.

## About Fiddler

Fiddler is the all-in-one AI Observability and Security platform for responsible AI. Monitoring and analytics capabilities provide a common language, centralized controls, and actionable insights to operationalize production ML models, GenAI, AI agents, and LLM applications with trust. An integral part of the platform, the Fiddler Trust Service provides quality and moderation controls for LLM applications. Powered by cost-effective, task-specific, and scalable Fiddler-developed trust models — including cloud and VPC deployments for secure environments — it delivers the fastest guardrails in the industry. Fortune 500 organizations utilize Fiddler to scale LLM and ML deployments, delivering high-performance AI, reducing costs, and ensuring responsible governance.

### Getting Started

1. Connect to Fiddler
2. Load a Data Sample
3. Define the Model Specifications
4. Set the Model Task
5. Create a Model
6. Set Up Alerts **(Optional)**
7. Create a Custom Metric **(Optional)**
8. Create a Segment **(Optional)**
9. Publish a Pre-production Baseline **(Optional)**
10. Configure a Rolling Baseline **(Optional)**
11. Publish Production Events

# 0. Imports

In [12]:
%pip install -q fiddler-client

import time as time

import pandas as pd
import fiddler as fdl

print(f'Running Fiddler Python client version {fdl.__version__}')

Note: you may need to restart the kernel to use updated packages.
Running Fiddler Python client version 3.10.0


## 1. Connect to Fiddler

Before you can add information about your model with Fiddler, you'll need to connect using the Fiddler Python client.


---


**We need a couple pieces of information to get started.**
1. The URL you're using to connect to Fiddler
2. Your authorization token

Your authorization token can be found by navigating to the **Credentials** tab on the **Settings** page of your Fiddler environment.

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()

URL = os.getenv('FIDDLER_URL')
TOKEN = os.getenv('FIDDLER_API_KEY')

Constants for this example notebook, change as needed to create your own versions

In [14]:
PROJECT_NAME = 'nickwong_quickstart_examples'  # If the project already exists, the notebook will create the model under the existing project.
MODEL_NAME = 'bank_churn_simple_monitoring'

STATIC_BASELINE_NAME = 'baseline_dataset'
ROLLING_BASELINE_NAME = 'rolling_baseline_1week'

# Sample data hosted on GitHub
PATH_TO_SAMPLE_CSV = 'https://media.githubusercontent.com/media/fiddler-labs/fiddler-examples/main/quickstart/data/v3/churn_data_sample.csv'
PATH_TO_EVENTS_CSV = 'https://media.githubusercontent.com/media/fiddler-labs/fiddler-examples/main/quickstart/data/v3/churn_production_data.csv'

Now just run the following to connect to your Fiddler environment.

In [15]:
fdl.init(url=URL, token=TOKEN)

260302T20:45:19.243Z     INFO| attached stderr handler to logger: auto_attach_log_handler=True, and root logger not configured 
260302T20:45:19.245Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/server-info GET -- emit req (0 B, timeout: (5, 15)) 
260302T20:45:19.519Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/server-info GET -- resp code: 200, took 0.273 s, resp/req body size: (1036 B, 0 B) 
260302T20:45:19.521Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/version-compatibility GET -- emit req (0 B, timeout: (5, 15)) 
260302T20:45:19.659Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/version-compatibility GET -- resp code: 200, took 0.137 s, resp/req body size: (2 B, 0 B) 


#### 1.a Create New or Load Existing Project

Once you connect, you can create a new project by specifying a unique project name in the fld.Project constructor and calling the `create()` method. If the project already exists, it will load it for use.

In [16]:
project = fdl.Project.get_or_create(name=PROJECT_NAME)

print(f'Using project with id = {project.id} and name = {project.name}')

260302T20:45:24.791Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/projects GET -- emit req (0 B, timeout: (5, 100)) 
260302T20:45:24.938Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/projects GET -- resp code: 200, took 0.147 s, resp/req body size: (160 B, 0 B) 
260302T20:45:24.939Z     INFO| Project not found, creating a new one - `nickwong_quickstart_examples` 
260302T20:45:24.939Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/projects POST -- emit req (40 B, timeout: (5, 100)) 
260302T20:45:25.048Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/projects POST -- resp code: 200, took 0.108 s, resp/req body size: (345 B, 40 B) 


Using project with id = 9b8ec5fd-928c-4943-aed2-c7b8e24c462e and name = nickwong_quickstart_examples


You should now be able to see the newly created project in the Fiddler UI.

<table>
    <tr>
        <td>
            <img src="https://media.githubusercontent.com/media/fiddler-labs/fiddler-examples/main/quickstart/images/simple_monitoring_1.png" />
        </td>
    </tr>
</table>

## 2. Load a Data Sample

In this example, we'll be considering the case where we're a bank and we have **a model that predicts churn for our customers**.
  
In order to get insights into the model's performance, **Fiddler needs a small sample of data** to learn the schema of incoming data.

In [17]:
sample_data_df = pd.read_csv(PATH_TO_SAMPLE_CSV)
column_list  = sample_data_df.columns
sample_data_df

,customer_id,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,predicted_churn,churn,timestamp
0,27acd1c2,545,Texas,Male,37,9,110483.86,1,1,1,127394.67,0.897202,yes,1710428231855
1,27b36d0c,497,Texas,Female,55,7,131778.66,1,1,1,9972.64,0.997441,yes,1710428262096
2,27b5360a,509,New York,Female,29,0,107712.57,2,1,1,92898.17,0.920563,yes,1710428292338
3,27b5d650,743,Hawaii,Nonbinary,39,6,0.00,2,1,0,44265.28,0.779282,yes,1710428322579
4,27b236a8,699,Florida,Female,25,8,0.00,2,1,1,52404.47,0.825474,yes,1710428352821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,27b409ba,686,Texas,Male,39,3,129626.19,2,1,1,103220.56,0.760645,yes,1711032910888
19996,27aaff96,446,Massachusetts,Female,45,10,125191.69,1,1,1,128260.86,0.216093,no,1711032941130
19997,27ad3162,794,California,Male,35,6,0.00,2,1,1,68730.91,0.982021,yes,1711032971371
19998,27b076ce,832,California,Male,61,2,0.00,1,0,1,127804.66,0.071598,no,1711033001613


## 3. Define the Model Specifications

In order to create a model in Fiddler, create a ModelSpec object with information about what each column of your data sample should used for.

Fiddler supports four column types:
1. **Inputs**
2. **Outputs** (Model predictions)
3. **Target** (Ground truth values)
4. **Metadata**

In [18]:
input_columns = list(
    column_list.drop(['predicted_churn', 'churn', 'customer_id', 'timestamp'])
)

In [19]:
model_spec = fdl.ModelSpec(
    inputs=input_columns,
    outputs=['predicted_churn'],
    targets=['churn'],  # Note: only a single Target column is allowed, use metadata columns and custom metrics for additional targets
    metadata=['customer_id', 'timestamp'],
)

If you have columns in your ModelSpec which denote **prediction IDs or timestamps**, then Fiddler can use these to power its analytics accordingly.

Let's call them out here and use them when configuring the Model in step 5.

In [21]:
id_column = 'customer_id'
timestamp_column = 'timestamp'

## 4. Set the Model Task

Fiddler supports a variety of model tasks. In this case, we're adding a binary classification model.

For this, we'll create a ModelTask object and an additional ModelTaskParams object to specify the ordering of our positive and negative labels.

*For a detailed breakdown of all supported model tasks, click [here](https://docs.fiddler.ai/technical-reference/python-client-guides/explainability/model-task-examples).*

In [22]:
model_task = fdl.ModelTask.BINARY_CLASSIFICATION

task_params = fdl.ModelTaskParams(target_class_order=['no', 'yes'])

## 5. Create a Model

Create a Model object and publish it to Fiddler, passing in
1. Your data sample
2. The ModelSpec object
3. The ModelTask and ModelTaskParams objects
4. The ID and timestamp columns

In [23]:
model = fdl.Model.from_data(
    name=MODEL_NAME,
    project_id=project.id,
    source=sample_data_df,
    spec=model_spec,
    task=model_task,
    task_params=task_params,
    event_id_col=id_column,
    event_ts_col=timestamp_column,
)

model.create()
print(f'New model created with id = {model.id} and name = {model.name}')

260302T20:48:57.077Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/files/upload POST -- emit req (0.917 MB, timeout: (120, 100)) 
260302T20:48:58.716Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/files/upload POST -- resp code: 200, took 1.639 s, resp/req body size: (507 B, 0.917 MB) 
260302T20:48:58.720Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/model-factory POST -- emit req (363 B, timeout: 600) 
260302T20:48:59.552Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/model-factory POST -- resp code: 200, took 0.831 s, resp/req body size: (4190 B, 363 B) 
260302T20:48:59.556Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/models POST -- emit req (4390 B, timeout: (5, 100)) 
260302T20:49:01.485Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/models POST -- resp code: 200, took 1.928 s, resp/req body size: (5573 B, 4390 B) 


New model created with id = 6cfd6c3e-8bdd-4ff0-a0f3-0aad095d8e48 and name = bank_churn_simple_monitoring


On the project page, you should now be able to see the newly onboarded model with its model schema.

<table>
    <tr>
        <td>
            <img src="https://github.com/fiddler-labs/fiddler-examples/blob/main/quickstart/images/simple_monitoring_3.png?raw=true" />
        </td>
    </tr>
</table>

<table>
    <tr>
        <td>
            <img src="https://github.com/fiddler-labs/fiddler-examples/blob/main/quickstart/images/simple_monitoring_4.png?raw=true" />
        </td>
    </tr>
</table>

## 6. Set Up Alerts (Optional)

Fiddler allows creating alerting rules when your data or model predictions deviate from expected behavior.

The alert rules can compare metrics to **absolute** or **relative** values.

Please refer to [our documentation](https://docs.fiddler.ai/technical-reference/python-client-guides/alerts-with-fiddler-client) for more information on Alert Rules.

---
  
Let's set up some alert rules.

The following API call sets up a Data Integrity type rule which triggers an email notification when published events have 2 or more range violations in any 1 day bin for the `numofproducts` column.

In [24]:
alert_rule_1 = fdl.AlertRule(
    name='Bank Churn Range Violation Alert',
    model_id=model.id,
    metric_id='range_violation_count',
    bin_size=fdl.BinSize.DAY,
    compare_to=fdl.CompareTo.RAW_VALUE,
    priority=fdl.Priority.HIGH,
    warning_threshold=2,
    critical_threshold=3,
    condition=fdl.AlertCondition.GREATER,
    columns=['numofproducts'],
)

alert_rule_1.create()
print(
    f'New alert rule created with id = {alert_rule_1.id} and name = {alert_rule_1.name}'
)

# Set notification configuration for the alert rule, a single email address for this simple example
alert_rule_1.set_notification_config(emails=['name@google.com'])

260302T20:51:32.443Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules POST -- emit req (456 B, timeout: (5, 100)) 
260302T20:51:32.678Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules POST -- resp code: 200, took 0.234 s, resp/req body size: (1343 B, 456 B) 
260302T20:51:32.680Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules/53ad04d9-dc06-4727-a9c1-90586ced44a6/notification PATCH -- emit req (31 B, timeout: (5, 100)) 


New alert rule created with id = 53ad04d9-dc06-4727-a9c1-90586ced44a6 and name = Bank Churn Range Violation Alert


260302T20:51:32.968Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules/53ad04d9-dc06-4727-a9c1-90586ced44a6/notification PATCH -- resp code: 200, took 0.287 s, resp/req body size: (149 B, 31 B) 


NotificationConfig(emails=['name@google.com'], pagerduty_services=[], pagerduty_severity='', webhooks=[])

Let's add a second alert rule.

This one sets up a Performance type rule which triggers an email notification when precision metric is 5% higher than that from 1 hr bin one day ago.

In [25]:
alert_rule_2 = fdl.AlertRule(
    name='Bank Churn Performance Alert',
    model_id=model.id,
    metric_id='precision',
    bin_size=fdl.BinSize.HOUR,
    compare_to=fdl.CompareTo.TIME_PERIOD,
    compare_bin_delta=24,  # Multiple of the bin size
    condition=fdl.AlertCondition.GREATER,
    warning_threshold=0.05,
    critical_threshold=0.1,
    priority=fdl.Priority.HIGH,
)

alert_rule_2.create()
print(
    f'New alert rule created with id = {alert_rule_2.id} and name = {alert_rule_2.name}'
)

# Set notification configuration for the alert rule, a single email address for this simple example
alert_rule_2.set_notification_config(emails=['name@google.com'])

260302T20:52:03.468Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules POST -- emit req (433 B, timeout: (5, 100)) 
260302T20:52:03.584Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules POST -- resp code: 200, took 0.115 s, resp/req body size: (1307 B, 433 B) 
260302T20:52:03.585Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules/98a53b52-a7d2-4ade-8238-e62a70065056/notification PATCH -- emit req (31 B, timeout: (5, 100)) 
260302T20:52:03.703Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/alert-rules/98a53b52-a7d2-4ade-8238-e62a70065056/notification PATCH -- resp code: 200, took 0.118 s, resp/req body size: (149 B, 31 B) 


New alert rule created with id = 98a53b52-a7d2-4ade-8238-e62a70065056 and name = Bank Churn Performance Alert


NotificationConfig(emails=['name@google.com'], pagerduty_services=[], pagerduty_severity='', webhooks=[])

## 7. Create a Custom Metric (Optional)

Fiddler's [Custom Metrics](https://docs.fiddler.ai/technical-reference/api-methods-30#custom-metrics) feature enables user-defined formulas for custom metrics.  Custom metrics will be tracked over time and can be used in Charts and Alerts just like the many out of the box metrics provided by Fiddler.  Custom metrics can also be managed in the Fiddler UI.

Please refer to [our documentation](https://docs.fiddler.ai/product-guide/monitoring-platform/custom-metrics) for more information on Custom Metrics.

---
  
Let's create an example custom metric.

In [26]:
custom_metric = fdl.CustomMetric(
    name='Lost Revenue',
    model_id=model.id,
    description='A metric to track revenue lost for each false positive prediction.',
    definition="""sum(if(fp(),1,0) * -100)""",  # This is an excel like formula which adds -$100 for each false positive predicted by the model
)

custom_metric.create()
print(
    f'New custom metric created with id = {custom_metric.id} and name = {custom_metric.name}'
)

260302T20:52:35.353Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/custom-metrics POST -- emit req (203 B, timeout: (5, 100)) 
260302T20:52:35.602Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/custom-metrics POST -- resp code: 200, took 0.248 s, resp/req body size: (909 B, 203 B) 


New custom metric created with id = eb1a5f21-2d67-4a71-92b5-fec85eab128a and name = Lost Revenue


## 8. Create a Segment (Optional)
Fiddler's [Segment API](https://docs.fiddler.ai/technical-reference/api-methods-30#segments) enables defining named cohorts/sub-segments in your production data. These segments can be tracked over time, added to charts, and alerted upon. Segments can also be managed in the Fiddler UI.

Please refer to our [documentation](https://docs.fiddler.ai/product-guide/monitoring-platform/segments) for more information on the creation and management of segments.

Let's create a segment to track customers from Hawaii for a specific age range.

In [27]:
segment = fdl.Segment(
    name='Hawaii Customers between 30 and 60',
    model_id=model.id,
    description='Hawaii Customers between 30 and 60',
    definition="(age<60 and age>30) and geography=='Hawaii'",
)

segment.create()
print(f'New segment created with id = {segment.id} and name = {segment.name}')

260302T20:53:48.149Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/segments POST -- emit req (212 B, timeout: (5, 100)) 
260302T20:53:48.408Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/segments POST -- resp code: 200, took 0.258 s, resp/req body size: (918 B, 212 B) 


New segment created with id = 2eca7b5b-f345-49dd-ae01-0f19861f8bcb and name = Hawaii Customers between 30 and 60


## 9. Publish a Static Baseline (Optional)

Since Fiddler already knows how to process data for your model, we can now add a **baseline dataset**.

You can think of this as a static dataset which represents **"golden data,"** or the kind of data your model expects to receive.

Then, once we start sending production data to Fiddler, you'll be able to see **drift scores** telling you whenever it starts to diverge from this static baseline.

***

Let's publish our **original data sample** as a pre-production dataset and then explicitly create a baseline from it.

**Note:** As of recent updates, baseline creation is now a separate, explicit step after uploading pre-production data. This gives you more control over when and how baselines are created.


*For more information on how to design your baseline dataset, [click here](https://docs.fiddler.ai/technical-reference/python-client-guides/publishing-production-data/creating-a-baseline-dataset).*

In [28]:
# Upload the pre-production data to make it available as a baseline
baseline_publish_job = model.publish(
    source=sample_data_df,
    environment=fdl.EnvType.PRE_PRODUCTION,
    dataset_name=STATIC_BASELINE_NAME,
)
print(
    f'Initiated pre-production environment data upload with Job ID = {baseline_publish_job.id}'
)

# Uncomment the line below to wait for the job to finish, otherwise it will run in the background.
# You can check the status on the Jobs page in the Fiddler UI or use the job ID to query the job status via the API.
# baseline_publish_job.wait()

260302T20:53:59.078Z     INFO| Model[bank_churn_simple_monitoring/v1] - Publishing events 
260302T20:53:59.093Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/files/upload POST -- emit req (0.720 MB, timeout: (120, 100)) 
260302T20:54:00.515Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/files/upload POST -- resp code: 200, took 1.422 s, resp/req body size: (514 B, 0.720 MB) 
260302T20:54:00.517Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/events POST -- emit req (193 B, timeout: (5, 100)) 
260302T20:54:00.992Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/events POST -- resp code: 202, took 0.475 s, resp/req body size: (166 B, 193 B) 
260302T20:54:00.993Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/jobs/860e04fb-e28a-4d2c-863c-f757b49b0654 GET -- emit req (0 B, timeout: (5, 100)) 
260302T20:54:01.127Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/jobs/860e04fb-e28a-4d2c-863c-f757b49b0654 GET -- resp code: 200, took 0.133 s, resp/req body size: (9

Initiated pre-production environment data upload with Job ID = 860e04fb-e28a-4d2c-863c-f757b49b0654


Now we need to **explicitly create a baseline** from the uploaded data. This step is required as automatic baseline creation has been removed from the pre-production data upload process.

## 10. Configure a Rolling Baseline (Optional)

Fiddler also allows you to configure a baseline based on **past production data.**

This means instead of looking at a static slice of data, it will look into past production events and use what it finds for drift calculation.

Please refer to [our documentation](https://docs.fiddler.ai/technical-reference/python-client-guides/publishing-production-data/creating-a-baseline-dataset) for more information on Baselines.

---
  
Let's set up a rolling baseline that will allow us to calculate drift relative to production data from 1 week back.

In [29]:
rolling_baseline = fdl.Baseline(
    model_id=model.id,
    name=ROLLING_BASELINE_NAME,
    type_=fdl.BaselineType.ROLLING,
    environment=fdl.EnvType.PRODUCTION,
    window_bin_size=fdl.WindowBinSize.DAY,  # Size of the sliding window
    offset_delta=7,  # How far back to set our window (multiple of window_bin_size)
)

rolling_baseline.create()
print(
    f'New rolling baseline created with id = {rolling_baseline.id} and name = {rolling_baseline.name}'
)

260302T20:56:19.878Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/baselines POST -- emit req (192 B, timeout: (5, 100)) 
260302T20:56:20.054Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/baselines POST -- resp code: 200, took 0.176 s, resp/req body size: (780 B, 192 B) 


New rolling baseline created with id = 0ce8c7df-5069-41b2-93f9-9070f1b1025f and name = rolling_baseline_1week


## 11. Publish Production Events

Finally, let's send in some production data!


Fiddler will **monitor this data and compare it to your baseline to generate powerful insights into how your model is behaving**.


---


Each record sent to Fiddler is called **an event**.
  
Let's load some sample events from a CSV file.

In [30]:
production_data_df = pd.read_csv(PATH_TO_EVENTS_CSV)

# Shift the timestamps of the production events to be as recent as today
production_data_df['timestamp'] = production_data_df['timestamp'] + (
    int(time.time() * 1000) - production_data_df['timestamp'].max()
)
production_data_df

,customer_id,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,predicted_churn,churn,timestamp
0,27c349a2,559,California,Male,52,2,0.00,1,1,0,129013.59,0.007448,no,1771880210519
1,27c35cee,482,California,Male,55,5,97318.25,1,0,1,78416.14,0.804852,yes,1771882639434
2,27c364f0,651,Florida,Female,46,4,89743.05,1,1,0,156425.57,0.012754,no,1771885068350
3,27c3627a,611,Hawaii,Male,38,7,0.00,1,1,1,63202.00,0.882252,yes,1771887497265
4,27c34164,696,California,Female,33,4,0.00,2,1,1,73371.65,0.999736,yes,1771889926181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,27c350b4,781,Hawaii,Female,48,0,57098.96,6,1,0,85644.06,0.032330,no,1772475294856
246,27c357e4,797,Hawaii,Female,55,10,0.00,9,1,1,49418.87,0.020316,no,1772477723772
247,27c36216,554,Hawaii,Male,31,1,0.00,7,0,1,192660.55,0.269628,yes,1772480152687
248,27c34d12,701,Hawaii,Nonbinary,37,1,0.00,7,1,0,163457.55,0.769625,yes,1772482581603


In [31]:
production_publish_job = model.publish(production_data_df)

print(
    f'Initiated production environment data upload with Job ID = {production_publish_job.id}'
)

# Uncomment the line below to wait for the job to finish, otherwise it will run in the background.
# You can check the status on the Jobs page in the Fiddler UI or use the job ID to query the job status via the API.
# production_publish_job.wait()

260302T20:57:48.142Z     INFO| Model[bank_churn_simple_monitoring/v1] - Publishing events 
260302T20:57:48.146Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/files/upload POST -- emit req (0.019 MB, timeout: (120, 100)) 
260302T20:57:48.381Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/files/upload POST -- resp code: 200, took 0.235 s, resp/req body size: (514 B, 0.019 MB) 
260302T20:57:48.382Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/events POST -- emit req (175 B, timeout: (5, 100)) 
260302T20:57:49.047Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/events POST -- resp code: 202, took 0.664 s, resp/req body size: (163 B, 175 B) 
260302T20:57:49.049Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/jobs/95518e4d-c828-4e47-a2b8-a33d29e5500e GET -- emit req (0 B, timeout: (5, 100)) 
260302T20:57:49.164Z     INFO| http: https://preprod.cloud.fiddler.ai/v3/jobs/95518e4d-c828-4e47-a2b8-a33d29e5500e GET -- resp code: 200, took 0.114 s, resp/req body size: (9

Initiated production environment data upload with Job ID = 95518e4d-c828-4e47-a2b8-a33d29e5500e


# Get Insights
  
Return to your Fiddler environment to get enhanced observability into your model's performance.

<table>
    <tr>
        <td>
            <img src="https://media.githubusercontent.com/media/fiddler-labs/fiddler-examples/main/quickstart/images/simple_monitoring_5.png" />
        </td>
    </tr>
</table>

**What's Next?**

Try the [LLM Monitoring - Quick Start Notebook](https://docs.fiddler.ai/tutorials-and-quick-starts/llm-and-genai/simple-llm-monitoring)

---


**Questions?**  
  
Check out [our docs](https://docs.fiddler.ai/) for a more detailed explanation of what Fiddler has to offer.